<a href="https://colab.research.google.com/github/zia207/High_Performance_Computing_R/blob/main/Notebook/02_01_03_hpc_bigdata_disk_frame_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1LhRCsnq8s3y0M76oN3-GEUfWQVL_yUv-)


# Using {disk.frame} for Large Datasets


`disk.frame` is an R package designed for handling large datasets that exceed available RAM by storing data on disk in a chunked format. It allows users to perform data manipulation using familiar dplyr syntax while processing data in parallel across multiple CPU cores. This tutorial demonstrates how to use disk.frame to work with the NYC taxi dataset, showcasing its capabilities for out-of-memory data manipulation. Note that disk.frame is soft-deprecated, and users are encouraged to consider the arrow package for new projects, but this tutorial focuses on disk.frame for educational purposes.


## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.


In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython


## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## disk.frame Installation Issues and Solutions

When attempting to install the [disk.frame](https://diskframe.com/) R package, users may encounter installation errors, especially on newer versions of R or certain operating systems. A common issue involves failed dependency installations (such as `fst` or `bit64`), or incompatibilities with recent CRAN policies. In some cases, disk.frame's dependencies require compilation with specific system libraries, or users may need to install an older version of the package that matches their R environment.

To fix these installation problems:
- Ensure all system dependencies required for compiling R packages are installed (for example, build tools on Windows or development libraries on Linux).
- Explicitly install any missing dependencies (`fst`, `bit64`, `dplyr`, `future`) before installing disk.frame.
- If a direct installation from CRAN fails, consider installing the development version from GitHub, which may include fixes for compatibility issues:


```r
# Install or update build tools if necessary (on Windows)
# install.packages("Rtools")

# Install disk.frame and dependencies from CRAN (preferred)
install.packages(c("fst", "bit64", "dplyr", "future")) # Install dependencies first
install.packages("disk.frame")

# If CRAN version fails, install the latest version from GitHub:
# install.packages("devtools")
# devtools::install_github("xiaodaigh/disk.frame")
```

With these steps, the disk.frame package should install successfully, allowing you to process large datasets efficiently in R. In this tutorial, we'll use disk.frame (properly installed and loaded as shown above) to work with the NYC taxi dataset, demonstrating out-of-memory data manipulation using familiar `dplyr` and `data.table`-like syntax.






## Check and Install Required R Packages


In [ ]:
%%R
packages <- c(
          'tidyverse',
          'data.table',
          'arrow',
          'disk.frame',
          'fst',
          'future'

)


In [ ]:
%%R
# Install missing packages
new.packages <- packages[!(packages %in% installed.packages(lib='drive/My Drive/R/')[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='drive/My Drive/R/')


### Verify Installation


In [ ]:
%%R
# Verify installation
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))


### Load Packages


In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
loaded_packages <- lapply(packages, function(pkg) {
  if (requireNamespace(pkg, quietly = TRUE)) {
    suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    TRUE
  } else {
    FALSE
  }
})
names(loaded_packages) <- packages

# If disk.frame is unavailable (it is soft-deprecated and may be missing),
# skip executable chunks so the tutorial still renders.
has_disk_frame <- isTRUE(loaded_packages[["disk.frame"]])
if (!has_disk_frame) {
  message("disk.frame is not available in this environment; skipping executable chunks.")
  knitr::opts_chunk$set(eval = FALSE)
}


### Check Loaded Packages


In [ ]:
%%R
# Check loaded packages
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])


### Data

We’ll use the 2023 Yellow Taxi Trip Data from NYC Open Data, which contains ~38 million rows with columns like `tpep_pickup_datetime`, `passenger_count`, `trip_distance`, `fare_amount`, `PULocationID`, `payment_type`, and more. We’ll load a subset (January 2023) to manage memory, using {arrow} to read the Parquet file and convert to {data.table}.


### Set Up disk.frame

Configure disk.frame to optimize performance. Set the number of chunks based on CPU cores and enable parallel processing with future.


In [ ]:
%%R
# Set up disk.frame
setup_disk.frame()


### Set number of chunks (e.g., 2x CPU cores for efficiency)


In [ ]:
%%R
ncores <- parallel::detectCores()
options(future.globals.maxSize = Inf) # Allow large data transfers
nchunks_optimal <- 2 * ncores
cat("Optimal number of chunks set to:",nchunks_optimal, "\n")


### Define data folder


In [ ]:
%%R
dataFolder <- "/home/zia207/WebSites/R_Website/High_Performance_Computing_R/R_Markdown/Data/"


### Load Data into disk.frame

Convert the CSV to a disk.frame object, which splits the data into fst files stored on disk. Use csv_to_disk.frame to read yellow_tripdata_2023.csv.


In [ ]:
%%R
# Define output folder for disk.frame chunks
df_path <- file.path(tempdir(), "yellow_tripdata_df")

# Convert CSV to disk.frame
df <- csv_to_disk.frame(
  infile = paste0(dataFolder, "yellow_tripdata_2023_01.csv"),
  outdir = df_path,
  header = TRUE,
  nchunks = 2 * ncores, # Number of chunks (adjust based on RAM)
  backend = "data.table" # Use data.table for fast reading
)

# Check disk.frame structure
print(df)


This creates a folder (df_path) containing chunked fst files, each representing a subset of the data.


## Basic Operations with dplyr Syntax

{disk.frame} supports {dplyr} verbs for data manipulation. Below, filter trips with a fare amount greater than $10 and select key columns.


### Filter and select


In [ ]:
%%R
# Filter and select
filtered_df <- df %>%
  filter(fare_amount > 10) %>%
  select(VendorID, tpep_pickup_datetime, fare_amount, passenger_count)

# View first few rows (collects to memory, use sparingly)
filtered_df %>% collect() %>% head()


The `collect()` function brings results into memory, so use it only for small outputs.


### Group-By and Aggregation

Perform aggregations, like calculating average fare by passenger count. disk.frame processes chunks in parallel.


In [ ]:
%%R
# Group by passenger_count and calculate mean fare
agg_df <- filtered_df %>%
  group_by(passenger_count) %>%
  summarise(avg_fare = mean(fare_amount, na.rm = TRUE))

# Collect results
agg_df %>% collect()


### Joining disk.frames

Join two disk.frame objects (e.g., filter subsets and join). Here, we create a subset for short trips and join with the filtered data.


In [ ]:
%%R
# Create a subset for trips with distance < 2 miles
short_trips_df <- df %>%
  filter(trip_distance < 2) %>%
  select(VendorID, tpep_pickup_datetime, trip_distance)

# Join with filtered_df
joined_df <- left_join(
  filtered_df,
  short_trips_df,
  by = c("VendorID", "tpep_pickup_datetime")
)

# Collect a sample
joined_df %>% collect() %>% head()


### Optimizing Performance

Use `srckeep` to load only necessary columns, reducing disk I/O. Example: keep only fare_amount and passenger_count.


In [ ]:
%%R
# Optimize by keeping only needed columns
df_opt <- df %>% srckeep(c("fare_amount", "passenger_count"))

# Verify columns
colnames(df_opt)


### Export Results

Export results to a CSV or convert to arrow for further use (recommended, as disk.frame is soft-deprecated).


In [ ]:
%%R
# export to CSV
output_csv <- file.path(tempdir(), "filtered_taxi_data.csv")
write_disk.frame(filtered_df, output_csv, overwrite = TRUE)
cat("Filtered data exported to:", output_csv, "\n")


## Clean Up

Remove temporary chunk files to free disk space.


In [ ]:
%%R
# Delete disk.frame folder
unlink(df_path, recursive = TRUE)


## Key Notes

- disk.frame is ideal for datasets (10-100 GB) too large for RAM but manageable on a single machine.
- Use chunk_size to balance memory and performance (smaller chunks for low RAM, larger for faster processing). d
- isk.frame is soft-deprecated; consider arrow for new projects (use as_arrow_dataset for conversion).
- Monitor disk space, as chunked fst files can grow large.
- Parallel processing requires sufficient CPU cores and proper future setup.

## Summary and Conclusion

This tutorial demonstrated using the {disk.frame} package in R to handle large datasets that exceed available RAM. By chunking data into fst files on disk, {disk.frame} enables efficient processing with familiar dplyr syntax. We covered setup, loading data, filtering, aggregation, joining, performance optimization, and exporting results. While {disk.frame} is useful for medium-sized datasets, it has been soft-deprecated in favor of the {arrow} package for new projects. Users should consider their specific needs and dataset sizes when choosing between these tools.

## References

1. ZJ D (2022). disk.frame: Larger-than-RAM Disk-Based Data Manipulation Framework. R package version 0.7.1, https://diskframe.com.

2. Wickham H, François R, Henry L, Müller K (2023). dplyr: A Grammar of Data Manipulation. R package version 1.1.2,  https://CRAN.R-project.org/package=dplyr.
